<div style="background:#C1ABA6">
<div style="font-size: xx-large ; font-weight: 900 ;  padding-top: 100px ; color: rgba(0 , 0 , 0 , 0.8); line-height: 100%"; align="center">Seizmogrami</div>
    
---
<div align="center"> Vježbe iz Seizmologije I </div>
<div align="center"> ak. god. 2026./2027. </div>
<div align="center"> dr.sc. Katarina Zailac </div>

---
</div>

# ObsPy

Koristit ćemo [ObsPy](https://docs.obspy.org/) biblioteku. [ObsPy](https://docs.obspy.org/) je projekt otvorenog koda koji pruža Python okvir za obradu seizmoloških podataka. On omogućuje:

* podršku za čitanje/pisanje datoteka različitih formata;
* pristup podatkovnim centrima;
* unificirane klase;
* funkcije za obradu signala.

Cilj projekta ObsPy je omogućiti brzi razvoj aplikacija za seizmologiju.

# Objekt tipa [`Stream`](https://docs.obspy.org/packages/autogen/obspy.core.stream.Stream.html#obspy.core.stream.Stream)

Seizmogrami u raznim formatima (npr. SAC, MiniSEED, GCF) se mogu učitati u objekt tipa [`Stream`](https://docs.obspy.org/packages/autogen/obspy.core.stream.Stream.html#obspy.core.stream.Stream) korištenjem funkcije [`read()`](https://docs.obspy.org/packages/autogen/obspy.core.stream.read.html).

Objekt tipa [`Stream`](https://docs.obspy.org/packages/autogen/obspy.core.stream.Stream.html#obspy.core.stream.Stream) je nalik listi, a sadrži više objekata tipa [`Trace`](https://docs.obspy.org/packages/autogen/obspy.core.trace.Trace.html#obspy.core.trace.Trace) koji pak sadrže neprekinuti vremenski niz podataka i pripadne meta-podatke (seizmogram).

Učitajmo jedan seizmogram i pogledajmo koje [`Trace`](https://docs.obspy.org/packages/autogen/obspy.core.trace.Trace.html#obspy.core.trace.Trace) objekte sadrži! Minimalno zadajemo ime datoteke te njezin format za učitavanje

```python
read(
    pathname_or_url=None,
    format=None,
    headonly=False,
    starttime=None,
    endtime=None,
    ...
    )
```

In [ ]:
import obspy
st = obspy.read("data/waveform_MOSL.mseed", format="mseed")
print(st)

[`Trace`](https://docs.obspy.org/packages/autogen/obspy.core.trace.Trace.html#obspy.core.trace.Trace) se jedinstveno identificira s kombinacijom:
* *network code* - registrirani identifikator mreže ili vlasnika podataka pri [FDSN-u](https://www.fdsn.org/)
* *station code* - oznaka postaje; u pravilu **nije jedinstvena** - uvijek je treba koristiti zajedno s kodom mreže da bi se osigurala jedinstvenost!
* *location code* - obično se koristi kako bi se logički odvojilo više instrumenata na istoj postaji
* *channel code* - kod od tri slova koji označava (prema [SEED standardu](https://epic.earthscope.org/webfm_send/2134))
    * *band* i frekvenciju uzorkovanja,
    * tip instrumenta
    * orijentaciju

### Učitavanje više seizmograma odjednom

* možemo koristiti UNIX *wildcard* da bismo učitali više seizmograma odjednom
* funkcija može automatski učitati više formata datoteka

In [ ]:
st = obspy.read("data/waveform_*")
print(st)

 Više [`Stream`](https://docs.obspy.org/packages/autogen/obspy.core.stream.Stream.html#obspy.core.stream.Stream) objekata se može kombinirati korištenjem operatora zbrajanja `+`.

In [ ]:
st1 = obspy.read("data/waveform_MOSL.mseed")
st2 = obspy.read("data/waveform_STON.mseed")

st = st1 + st2
print("Seizmogrami postaja MOSL i STON")
print(st)

st3 = obspy.read("data/waveform_KALN.mseed")
st += st3
print("\nNakon dodavanja seizmograma postaje KALN")
print(st)

Budući da je [`Stream`](https://docs.obspy.org/packages/autogen/obspy.core.stream.Stream.html#obspy.core.stream.Stream) tehnički [`iterable`](https://realpython.com/ref/glossary/iterable/) objekt, može se korisno iterirati po elementima, što su [`Trace`](https://docs.obspy.org/packages/autogen/obspy.core.trace.Trace.html#obspy.core.trace.Trace) objekti.

In [ ]:
for tr in st:
    print(tr.id)

# Objekt tipa [`Trace`](https://docs.obspy.org/packages/autogen/obspy.core.trace.Trace.html#obspy.core.trace.Trace)

* [`Trace`](https://docs.obspy.org/packages/autogen/obspy.core.trace.Trace.html#obspy.core.trace.Trace) je neprekinuti vremenski niz podataka 
* sadrži i pripadne metapodatke koji u potpunosti opisuju podatke
    * lokacija (kodovi mreže, postaje, lokacije, kanala)
    * početno vrijeme zapisa
    * frekvencija uzorkovanja

In [ ]:
st = obspy.read("data/waveform_MOSL.mseed")
tr = st[0]  # ovime izvlacimo prvi trace u streamu

# metapodaci su sadrzani u objektu tr.stats
print(tr.stats)

# Prikaz seizmograma

Za prikaz samog zapisa može se koristiti metoda [`plot()`](https://docs.obspy.org/packages/autogen/obspy.core.stream.Stream.plot.html#obspy.core.stream.Stream.plot) koja će kreirati sliku učitanog seizmograma.

In [ ]:
st.plot();

Slično se može nacrtati i samo jedan *trace*.

In [ ]:
tr.plot();

Ako želimo izrezati samo onaj dio seizmograma koji nam je zanimljiv, možemo koristiti metodu [`trim()`](https://docs.obspy.org/packages/autogen/obspy.core.stream.Stream.trim.html#obspy.core.stream.Stream.trim).

<div class="alert alert-warning rounded-pill rounded-5" style="margin: auto auto 10px auto; text-align: center;">
    <strong>Napomena</strong>: <a https://docs.obspy.org/packages/autogen/obspy.core.stream.Stream.html><code>Stream</code></a> i <a https://docs.obspy.org/packages/autogen/obspy.core.trace.Trace.html><code>Trace</code></a> objekti su <i>mutable</i> objekti što znači da se prilikom mijenjanja nečega kod tog objekta mijenja inicijalni objekt (kao što su <i>mutable</i> i Python liste, na primjer). Stoga kod operacija nad [<a https://docs.obspy.org/packages/autogen/obspy.core.stream.Stream.html><code>Stream</code></a> i <a https://docs.obspy.org/packages/autogen/obspy.core.trace.Trace.html><code>Trace</code></a> objektima, ako želimo da nam originalna varijabla ostane nepromijenjena, koristimo metodu <a https://docs.obspy.org/packages/autogen/obspy.core.trace.Trace.copy.html><code>copy()</code></a>.
</div>

In [ ]:
print(tr);
tr_trimmed = tr.copy()
tr_trimmed.trim(tr.stats.starttime + 24 * 60, tr.stats.starttime + 26 * 60)
print(tr_trimmed);
tr_trimmed.plot();

Postoje i metode za uklanjanje trendova, filtriranje ([`filter()`](https://docs.obspy.org/packages/autogen/obspy.core.trace.Trace.filter.html#obspy.core.trace.Trace.filter))...

In [ ]:
tr_filtered = tr_trimmed.copy()
tr_filtered.detrend("linear")
tr_filtered.taper(max_percentage=0.05, type='cosine')
tr_filtered.filter("bandpass", freqmin=3, freqmax=6, corners=2, zerophase=True)
tr_filtered.plot();

# "Sirovi" podaci

Sami podaci su spremljeni u obliku [NumPy ndarraya](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.html) kao `Trace.data`.

In [ ]:
print(tr.data)
print(len(tr.data))

## Izrada posebnih grafova pomoću knjižnice [`matplotlib`](https://matplotlib.org/)

Ako bismo htjeli napraviti poseban graf, i nismo zadovoljni s ograničenim mogućnostima implementirane metode [`plot()`](https://docs.obspy.org/packages/autogen/obspy.core.stream.Stream.plot.html), možemo direktno pristupiti "sirovim" podacima i nacrtati ih korištenjem dobro poznate knjižnice [`matplotlib`](https://matplotlib.org/). 

Treba voditi računa o prikazima datuma. Knjižnica [`matplotlib`](https://matplotlib.org/) i [`ObsPy`](https://github.com/obspy/obspy/wiki/) različito definiraju datume. Stoga je prilikom korištenja vremena (metoda [`times()`](https://docs.obspy.org/packages/autogen/obspy.core.trace.Trace.times.html)) potrebno koristiti argument `type="matplotlib"`. 

In [ ]:
import matplotlib.pyplot as plt

fig = plt.figure()
ax = fig.add_subplot(1, 1, 1)
ax.plot(tr_filtered.times(type="matplotlib"), tr_filtered.data, "r-")
ax.xaxis_date()  # govorimo matplotlibu da su nam na x-osi datumi
fig.autofmt_xdate()  # automatski organizira datume na x-osi (da se ne preklapaju, npr)
plt.show();

# Sintetski seizmogrami

Podaci se mogu i sintetizirati u formi [NumPy ndarray-a](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.html), spremiti u [`Trace`](https://docs.obspy.org/packages/autogen/obspy.core.trace.Trace.html) objekt. Potrebno je zadati i određene metapodatke, te se onda [`Trace`](https://docs.obspy.org/packages/autogen/obspy.core.trace.Trace.html) može koristiti kao i ranije.

In [ ]:
from obspy import Trace, UTCDateTime
import numpy as np

x = np.random.randint(-100, 100, 500)
tr = Trace(data=x)
tr.stats.station = "XYZ"
tr.stats.starttime = UTCDateTime()

tr.plot();

# Zadaci

## Zadatak 1

Kreirajte `numpy.ndarray` s nulama (npr. koristite [`numpy.zeros()`](https://numpy.org/doc/stable/reference/generated/numpy.zeros.html) i kreirajte idealni puls negdje u njemu.


## Zadatak 2

Inicijalizirajte [`Trace`](https://docs.obspy.org/packages/autogen/obspy.core.trace.Trace.html) objekt sa sintetiziranim podacima. Upišite neke od metapodataka (npr. kod mreže, postaje, ...), ispišite info te prikažite seizmogram.

## Zadatak 3

Koristite metodu [`Trace.filter`](https://docs.obspy.org/packages/autogen/obspy.core.trace.Trace.filter.html), te filtrirajte Vaš trace *lowpass* filtrom s graničnom frekvencijom od 1 Hz. Prikažite filtrirani seizmogram.

## Zadatak 4

Skalirajte Vaše seizmograme faktorom 500 te dodajte standardni Gaussijanski šum (npr. koristite [`numpy.random.rand`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.rand.html). Prikažite seizmogram.

## Zadatak 5

Učitajte sve seizmograme Zagrebačkog potresa (direktorij [`data/20200322`](data/20200322)). Provjerite koliko ima zapisa. Odaberite zapise iz CR mreže (npr. korištenjem [`Stream.select`](https://docs.obspy.org/packages/autogen/obspy.core.stream.Stream.select.html) te ih nacrtajte.

## Zadatak 6

Filtrirajte sve odabrane zapise *bandpass* filtrom između frekvencija 4 i 8 Hz. Prikažite filtrirane zapise.